# Comprensión y análisis exploratorio de datos (EDA)

**Dataset:** riesgo crediticio (10,763 registros, 23 columnas).
**Variable objetivo:** `Pago_atiempo` (asumida como objetivo; pendiente de confirmación del instructor).

**Objetivo del notebook:** entender la estructura y calidad de los datos, la relación de cada variable con el objetivo, y derivar de allí (1) reglas de validación de datos, (2) transformaciones candidatas y (3) atributos derivados, que serán usados en etapas posteriores del proyecto.

## 1. Exploración inicial

### 1.1 Carga y descripción general

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Ubica la raíz del repo sin importar desde dónde arranque el kernel
raiz = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_excel(raiz / "data" / "raw" / "base_datos.xlsx")

print(df.shape)
df.head()

In [ ]:
df.info()

**Comentario.** 10,763 filas y 23 columnas. Varias columnas de dinero aparecen como `float64` no porque tengan decimales sino porque contienen nulos: el tipo `int64` de pandas no admite `NaN`, así que cualquier columna entera con faltantes se convierte automáticamente a flotante. No es un error de los datos sino un artefacto del manejo de nulos.

### 1.2 Caracterización de variables (tipo, escala, naturaleza)

In [ ]:
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1)
})
resumen

**Clasificación de las 23 variables.** El `dtype` no basta para caracterizar: hay códigos categóricos almacenados como enteros. Clasificación según la naturaleza de cada variable:

| Grupo | Variables | Observaciones |
|---|---|---|
| **Identificación temporal** | `fecha_prestamo` | Fecha de originación. No es predictora directa; sirve para análisis de cosechas (vintage). |
| **Categóricas nominales politómicas** | `tipo_credito` | Código de producto. Viene como `int64` pero es un identificador: promediarla no tiene sentido. |
| **Categóricas dicotómicas** | `tipo_laboral`, `Pago_atiempo` (objetivo) | `tipo_laboral` tiene exactamente 2 categorías (dicotómica, no politómica). |
| **Categórica ordinal (contaminada)** | `tendencia_ingresos` | En teoría ordinal (dirección de la tendencia), pero llega como `object` con 46 valores únicos porque mezcla las categorías válidas con valores numéricos corruptos. Se limpia en 1.5. |
| **Numéricas continuas** | `capital_prestado`, `salario_cliente`, `cuota_pactada`, `total_otros_prestamos`, `saldo_mora`, `saldo_total`, `saldo_principal`, `saldo_mora_codeudor`, `promedio_ingresos_datacredito`, `puntaje`, `puntaje_datacredito` | Montos en pesos y puntajes de riesgo. |
| **Numéricas discretas (conteos)** | `plazo_meses`, `edad_cliente`, `cant_creditosvigentes`, `huella_consulta`, `creditos_sectorFinanciero`, `creditos_sectorCooperativo`, `creditos_sectorReal` | Enteros no negativos. Los conteos de créditos por sector describen el perfil de endeudamiento. |


### 1.3 Revisión y unificación de nulos

Los nulos aparecen en dos formas: `NaN` explícitos y **valores centinela** (números imposibles que codifican "sin dato"). Ambos deben unificarse a `NaN` para que el conteo de faltantes sea real.

In [ ]:
# Nulos explícitos
df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)

In [ ]:
# ¿promedio_ingresos_datacredito y tendencia_ingresos faltan en las mismas filas?
pd.crosstab(df["promedio_ingresos_datacredito"].isna(),
            df["tendencia_ingresos"].isna(),
            rownames=["promedio_ingresos NaN"], colnames=["tendencia NaN"])

**Comentario.** Las dos variables con ~27% de faltantes (`promedio_ingresos_datacredito` y `tendencia_ingresos`) faltan **en las mismas filas**: provienen del mismo módulo de la central de riesgo, y cuando la central no reporta ese módulo, faltan las dos a la vez. El mecanismo del faltante es por bloque, no aleatorio por celda.

**¿El faltante es informativo?** Se probó una bandera de faltante contra el objetivo: si a los clientes sin este módulo les fuera peor (o mejor), la bandera discriminaría. No lo hace (diferencia despreciable en la tasa de pago), lo que sugiere un **faltante no informativo** respecto del objetivo. Verificación:

In [ ]:
flag = df["promedio_ingresos_datacredito"].isna()
df.groupby(flag)["Pago_atiempo"].agg(["mean", "size"]).rename(
    columns={"mean": "tasa_pago", "size": "n"})

In [ ]:
# Centinelas en edad_cliente: edades imposibles (>=120) codifican "sin dato"
print(df.loc[df["edad_cliente"] >= 120, "edad_cliente"].value_counts())
print("Total centinelas de edad:", (df["edad_cliente"] >= 120).sum())

In [ ]:
# Centinela en puntaje_datacredito: 0 no es un puntaje real de central de riesgo
print("Registros con puntaje_datacredito == 0:", (df["puntaje_datacredito"] == 0).sum())
df["puntaje_datacredito"].describe()

In [ ]:
# saldo_mora_codeudor: pocos valores únicos y nulos probablemente estructurales
df["saldo_mora_codeudor"].value_counts(dropna=False).head(10)

**Decisiones de unificación de nulos:**

1. `edad_cliente` en 121–123 → `NaN`. Son centinelas de "edad no registrada", **no** outliers reales: tratarlos como edades verdaderas distorsionaría toda la distribución.
2. `puntaje_datacredito == 0` → `NaN`. Un 0 no es un puntaje emitido por la central; codifica ausencia de puntaje.
3. `saldo_mora_codeudor` nulo → `0`. Hipótesis: el nulo significa "el crédito no tiene codeudor", es decir es un **nulo estructural** con significado propio, no un dato perdido. Se imputa 0 (sin mora de codeudor) y se deja documentado como supuesto a validar con el negocio.
4. `saldo_mora`, `saldo_total`, `saldo_principal`: se dejan como `NaN` (clientes sin historial reportado); la estrategia de imputación se decide en la etapa de preparación, no aquí.

In [ ]:
df.loc[df["edad_cliente"] >= 120, "edad_cliente"] = np.nan
df.loc[df["puntaje_datacredito"] == 0, "puntaje_datacredito"] = np.nan
df["saldo_mora_codeudor"] = df["saldo_mora_codeudor"].fillna(0)

df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)

### 1.4 Eliminación de variables irrelevantes

In [ ]:
# puntaje: ¿cuánta variabilidad tiene realmente?
df["puntaje"].value_counts(normalize=True).head(5).round(4)

**Comentario.** `puntaje` es casi una constante: cerca del **87%** de las filas comparten el mismo valor. Una variable casi constante no puede discriminar el objetivo (no hay contraste que explotar) y se elimina.

**Sobre `tendencia_ingresos`:** aunque tiene 27% de faltantes y señal débil, se **conserva** por ahora: en el análisis bivariable (sección 3) se muestra que su relación con el objetivo, aunque de magnitud pequeña, es monotónica en el sentido esperado. La decisión de excluirla del modelo se pospone a la etapa de selección de variables, donde competirá con las demás. Eliminar solo por porcentaje de nulos sería descartar información sin evaluar si el faltante es informativo (ya se mostró que no) ni si la parte observada aporta (aporta poco, pero en la dirección correcta).

In [ ]:
df = df.drop(columns=["puntaje"])
df.shape

### 1.5 Corrección de tipos de datos

`tendencia_ingresos` mezcla categorías de texto con valores numéricos corruptos; se separan y los corruptos pasan a `NaN`. Los códigos categóricos pasan a tipo `category`.

In [ ]:
tend_raw = df["tendencia_ingresos"]
es_numero = pd.to_numeric(tend_raw, errors="coerce").notna()

print(f"Valores numericos corruptos dentro de la columna: {es_numero.sum()}")
print("\nCategorias validas (texto):")
print(tend_raw[~es_numero].value_counts(dropna=False))

In [ ]:
# Los valores numéricos corruptos se tratan como nulos
df["tendencia_ingresos"] = tend_raw.where(~es_numero)

# Conversión de tipos
df["tipo_credito"] = df["tipo_credito"].astype("category")
df["tipo_laboral"] = df["tipo_laboral"].astype("category")
df["tendencia_ingresos"] = df["tendencia_ingresos"].astype("category")
df["Pago_atiempo"] = df["Pago_atiempo"].astype(int)  # 0/1

df.dtypes

**Comentario.** Tras la limpieza, cada columna tiene un tipo uniforme y coherente con su naturaleza: los códigos son `category`, los montos y conteos son numéricos, la fecha es `datetime`. La contaminación de `tendencia_ingresos` (números dentro de una columna categórica) es en sí misma un hallazgo de calidad de datos que alimenta las reglas de validación de la sección 5.

## 2. Análisis univariable

### 2.1 Variable objetivo

In [ ]:
tabla_obj = df["Pago_atiempo"].value_counts()
print(tabla_obj)
print()
print((df["Pago_atiempo"].value_counts(normalize=True) * 100).round(2))

sns.countplot(x="Pago_atiempo", data=df)
plt.title("Distribución de la variable objetivo Pago_atiempo")
plt.show()

**Comentario — desbalance severo.** Aproximadamente **95% de los créditos se pagaron a tiempo y solo ~5% no** (razón cercana a 20:1). Consecuencias directas para el resto del proyecto:

- El **accuracy es inútil** como métrica: un modelo que prediga "todos pagan" acierta ~95% sin aprender nada. Es el mismo fenómeno del valor predictivo en enfermedades de baja prevalencia: con prevalencia del 5%, la exactitud global esconde el desempeño sobre la clase que interesa.
- Las métricas relevantes serán recall/precisión sobre la clase minoritaria, F1, AUC-ROC y AUC-PR.
- En entrenamiento habrá que considerar técnicas para desbalance (ponderación de clases, submuestreo/sobremuestreo).

### 2.2 Variables numéricas

In [ ]:
numericas = df.select_dtypes(include=[np.number]).columns.drop("Pago_atiempo")
df[numericas].describe().T.round(2)

In [ ]:
# Medidas de forma: asimetría (skewness) y curtosis
pd.DataFrame({
    "skewness": df[numericas].skew(),
    "kurtosis": df[numericas].kurtosis()
}).round(2).sort_values("skewness", ascending=False)

In [ ]:
df[numericas].hist(figsize=(16, 14), bins=30)
plt.suptitle("Histogramas de variables numéricas", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), numericas):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col, fontsize=9)
for ax in axes.ravel()[len(numericas):]:
    ax.set_visible(False)
plt.suptitle("Boxplots de variables numéricas", y=1.01)
plt.tight_layout()
plt.show()

**Comentario.**

- Las variables de **dinero** (`capital_prestado`, `salario_cliente`, `cuota_pactada`, saldos, `promedio_ingresos_datacredito`) muestran **fuerte asimetría positiva** (skewness alta, colas largas a la derecha): muchos valores moderados y pocos muy grandes. Es el patrón típico de variables monetarias; ninguna es gaussiana. Candidatas naturales a transformación logarítmica.
- Los **conteos** (`cant_creditosvigentes`, `huella_consulta`, créditos por sector) también son asimétricos, concentrados en valores bajos.
- `edad_cliente` (ya sin centinelas) y `puntaje_datacredito` (ya sin ceros) tienen distribuciones mucho más razonables; el puntaje se concentra en el rango típico de las centrales de riesgo.
- Los boxplots marcan muchos puntos como "outliers", pero en variables log-normales eso es esperable: **no** se eliminan mecánicamente; se documentan y se decidirá con la transformación.

### 2.3 Variables categóricas

In [ ]:
categoricas = ["tipo_credito", "tipo_laboral", "tendencia_ingresos"]
for col in categoricas:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, categoricas):
    sns.countplot(x=col, data=df, ax=ax,
                  order=df[col].value_counts().index)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Comentario.** `tipo_credito` concentra el volumen en pocos productos y tiene colas de productos con muy pocos registros: al modelar, las categorías raras tendrán estimaciones inestables (candidatas a agruparse en "otros"). `tipo_laboral` es dicotómica con ambos grupos bien representados. En `tendencia_ingresos`, además del 27% estructuralmente faltante, la limpieza de valores corruptos aumentó los nulos.

## 3. Análisis bivariable (cada variable vs. objetivo)

La pregunta en cada caso: ¿la distribución de la variable difiere entre quienes pagan a tiempo y quienes no? Método: hipótesis → magnitud → dirección → segmentación.

In [ ]:
# Numéricas: mediana por clase del objetivo
comparacion = df.groupby("Pago_atiempo")[numericas].median().T
comparacion.columns = ["No pagó a tiempo (0)", "Pagó a tiempo (1)"]
comparacion.round(1)

In [ ]:
claves = ["capital_prestado", "cuota_pactada", "salario_cliente",
          "edad_cliente", "puntaje_datacredito", "plazo_meses"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), claves):
    sns.boxplot(x="Pago_atiempo", y=col, data=df, ax=ax, showfliers=False)
    ax.set_title(col)
plt.suptitle("Variables numéricas clave según Pago_atiempo (sin outliers para legibilidad)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categóricas: tasa de pago por categoría
for col in categoricas:
    tabla = df.groupby(col, observed=True)["Pago_atiempo"].agg(
        tasa_pago="mean", n="size").sort_values("tasa_pago")
    tabla["tasa_pago"] = (tabla["tasa_pago"] * 100).round(2)
    print(f"--- {col} ---")
    print(tabla)
    print()

**Comentario.**

- Ninguna variable individual separa las clases de forma dramática — coherente con un problema de riesgo de crédito real, donde la señal está repartida en muchas variables débiles.
- `puntaje_datacredito` muestra la dirección esperada: puntajes más bajos entre quienes no pagaron a tiempo. Es de las variables individualmente más prometedoras.
- `tipo_credito` presenta diferencias de tasa de pago entre productos: el producto **confunde** otras relaciones (los productos difieren en plazo, monto y perfil de cliente a la vez). Esto obliga a segmentar por producto antes de interpretar cualquier relación cruda.
- `tendencia_ingresos`: la tasa de pago varía en forma **monotónica** entre categorías pero con un rango total de apenas ~2–3 puntos porcentuales — señal real pero débil, como se anticipó en 1.4.
- Diferencias en montos (`capital_prestado`, `cuota_pactada`) entre clases son visibles pero moderadas y probablemente mediadas por el tipo de producto.

## 4. Análisis multivariable

### 4.1 Correlaciones entre numéricas

In [ ]:
plt.figure(figsize=(13, 10))
corr = df[numericas].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7})
plt.title("Matriz de correlación (variables numéricas)")
plt.show()

**Comentario.** Se observan bloques de correlación esperables: (1) los montos del crédito entre sí — `capital_prestado`, `cuota_pactada` y `plazo_meses` están ligados por la aritmética del crédito (la cuota es función del capital y el plazo), por lo que aportan información parcialmente redundante y habrá que vigilar colinealidad en modelos lineales; (2) los saldos reportados por la central entre sí; (3) los conteos de créditos con `cant_creditosvigentes`. La correlación de cada predictor con `Pago_atiempo` es individualmente baja, consistente con el bivariable.

In [ ]:
muestra = df.sample(n=min(2000, len(df)), random_state=42)
sns.scatterplot(data=muestra, x="capital_prestado", y="cuota_pactada",
                hue="Pago_atiempo", alpha=0.4, s=15)
plt.title("Capital vs cuota pactada, coloreado por Pago_atiempo (muestra n=2000)")
plt.show()

In [ ]:
pd.crosstab(df["tipo_credito"], df["tipo_laboral"], normalize="index").round(3)

### 4.2 Análisis temporal: ¿existe un efecto de cosecha (vintage)?

La tasa de pago por mes de originación aparenta variar en el tiempo. Antes de crear variables temporales derivadas hay que descartar que sea **confusión por mezcla de productos**: si en ciertos meses se colocaron créditos de plazos/productos distintos, la variación mensual reflejaría la mezcla, no un deterioro temporal real.

In [ ]:
from scipy.stats import spearmanr

df["mes"] = df["fecha_prestamo"].dt.to_period("M")
mensual = df.groupby("mes").agg(
    tasa_pago=("Pago_atiempo", "mean"),
    plazo_mediano=("plazo_meses", "median"),
    n=("Pago_atiempo", "size"))

fig, ax1 = plt.subplots(figsize=(13, 4))
mensual["tasa_pago"].plot(ax=ax1, marker="o", label="tasa de pago")
ax1.set_ylabel("tasa de pago")
ax2 = ax1.twinx()
mensual["plazo_mediano"].plot(ax=ax2, color="orange", marker="s",
                              alpha=0.6, label="plazo mediano")
ax2.set_ylabel("plazo mediano (meses)")
plt.title("Tasa de pago mensual vs plazo mediano de los créditos colocados")
plt.show()

rho_plazo, p1 = spearmanr(mensual["plazo_mediano"], mensual["tasa_pago"])
rho_n, p2 = spearmanr(mensual["n"], mensual["tasa_pago"])
print(f"Spearman tasa_pago ~ plazo_mediano: rho = {rho_plazo:.3f} (p = {p1:.4f})")
print(f"Spearman tasa_pago ~ volumen (n):  rho = {rho_n:.3f} (p = {p2:.4f})")

**Comentario.** La correlación de la tasa de pago mensual con el **plazo mediano** de los créditos colocados ese mes es claramente negativa (ρ ≈ −0.45), mientras que con el **volumen** colocado es casi nula (ρ ≈ 0.07). Interpretación: los meses con peor tasa de pago son meses donde la mezcla de productos/plazos fue distinta, no evidencia de un deterioro temporal genuino. Es un **confusor clásico**: la fecha se asocia al desenlace a través de la mezcla de producto. **Decisión:** se descartan las variables temporales derivadas (mes, cosecha, antigüedad) como predictoras; `fecha_prestamo` queda solo como metadato.

In [ ]:
df = df.drop(columns=["mes"])  # columna auxiliar del análisis, no se conserva

## 5. Conclusiones

### 5.1 Reglas de validación de datos (para la etapa de ingesta/preparación)

Derivadas de los hallazgos de calidad de este EDA. Todo registro nuevo debería validarse contra:

1. `edad_cliente` en rango [18, 100]; valores ≥ 120 son centinelas → tratar como nulo.
2. `puntaje_datacredito` > 0 o nulo; el 0 es centinela de "sin puntaje".
3. `tendencia_ingresos` debe pertenecer al conjunto de categorías válidas de texto; cualquier valor numérico en esta columna es corrupción → nulo.
4. Montos (`capital_prestado`, `cuota_pactada`, `salario_cliente`, saldos) ≥ 0.
5. `Pago_atiempo` ∈ {0, 1}.
6. `tipo_laboral` con exactamente las 2 categorías conocidas; `tipo_credito` dentro del catálogo de productos.
7. Si falta `promedio_ingresos_datacredito`, se espera que falte también `tendencia_ingresos` (faltan por bloque); una sola de las dos faltando es anomalía a revisar.
8. `saldo_mora_codeudor` nulo se interpreta como "sin codeudor" → 0 (supuesto a validar con negocio).

### 5.2 Transformaciones propuestas

- **Log-transformación** (log1p) de las variables monetarias por su fuerte asimetría positiva.
- **Agrupación de categorías raras** de `tipo_credito` en "otros" para estabilidad de estimación.
- **Imputación** de los saldos de central de riesgo: decidir entre mediana + bandera de faltante, o imputación por modelo, en la etapa de preparación. Para el bloque de ingresos (27%), el faltante mostró ser no informativo, lo que habilita imputación simple sin sesgo aparente.
- **Ponderación o remuestreo** por el desbalance 20:1 del objetivo.
- Escalado estándar/robusto para modelos sensibles a escala.

### 5.3 Atributos derivados candidatos (y descartados)

**Candidatos a probar en feature engineering:**
- `cuota_pactada / salario_cliente`: proxy de capacidad de pago (carga de la cuota sobre el ingreso).
- `saldo_mora / saldo_total`: proporción del endeudamiento en mora.
- `total_creditos_sectores = creditos_sectorFinanciero + creditos_sectorCooperativo + creditos_sectorReal`: exposición total.

**Probados y descartados empíricamente en este EDA:**
- Bandera de faltante del bloque de ingresos: no discrimina el objetivo (1.3).
- Variables temporales derivadas (mes/cosecha): el efecto aparente es confusión por mezcla de productos (4.2).

**Nota final.** La señal individual de todas las variables es débil y el objetivo está severamente desbalanceado: el valor del modelo vendrá de la combinación de muchas variables débiles y de una elección de métricas adecuada a la baja prevalencia, no de un predictor dominante.